# 云计算技术 · 期中大作业（**学生版**）
## 覆盖范围：第 1 – 8 讲

---
**姓名**：___程泽凯________　　**学号**：__2023112621___　　**作业提交日期**：___2026年5月11日______________

---

### 考试说明

- **答题方式**：将代码/YAML 中所有 `???` 替换为正确内容；不要修改函数名和参数
- **执行顺序**：按顺序从上到下执行每一个单元格
- **提交要求**：完成后另存为 `学号_姓名_midterm.ipynb` 发送至助教邮箱，截止时间为5月15日晚上10:00，如延期提交将按天扣除分数（延期一天扣10分）。

### 分值分布

| 任务 | 涉及讲次 | 内容 | 分值 |
|------|---------|------|------|
| **任务一** | 第 1-2 讲 | Docker 容器生命周期管理（docker-py） | **30 分** |
| **任务二** | 第 3-5 讲 | Kubernetes Deployment + Service 编写与部署 | **30 分** |
| **任务三** | 第 6-7 讲 | Redis 数据结构操作 | **40 分** |
| 加分项 A | 第 8 讲 | MinIO 对象存储操作 | **+5 分** |
| 加分项 B | 第 3 讲 | Docker Compose 多容器编排 YAML | **+3 分** |
| 加分项 C | 第 3 讲 | 熔断器（Circuit Breaker）模式实现 | **+2 分** |

> ⚡ **重要提示**：
> 1. 任务一会启动 Redis 容器，**任务三依赖任务一**，请先完成任务一
> 2. 任务二独立（依赖 Minikube），可与任务一、三并行
> 3. 遇到难点可先跳过，优先完成后续可得分部分
> 4. 所有填空处均有 **中文注释** 给出具体 API 签名与示例

---
## 模块 0：一键环境配置

运行下方单元格自动完成：Python 依赖安装 → Docker 检测 → Minikube 集群检测。
**预计耗时 1-2 分钟**。

In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  模块 0：环境自动配置（无需修改）                              ║
# ║  安装 Python 依赖 + 检测 Docker / Minikube 集群状态            ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess
import sys
import platform
import shutil
import time
import os
import socket

OS = platform.system()


def run(cmd, check=False, timeout=None):
    """
    执行 shell 命令，返回 (stdout, stderr, returncode)

    解决 Windows + Jupyter 中 subprocess 读取 minikube/kubectl 输出时
    出现 UnicodeDecodeError: 'gbk' codec can't decode byte 的问题。
    """
    try:
        r = subprocess.run(
            cmd,
            shell=True,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout
        )

        if check and r.returncode != 0:
            raise RuntimeError(r.stderr.strip())

        return r.stdout.strip(), r.stderr.strip(), r.returncode

    except subprocess.TimeoutExpired:
        return "", f"命令执行超时：{cmd}", 124


def pip_install(*pkgs):
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            *pkgs,
            "-q",
            "--disable-pip-version-check"
        ],
        stdout=subprocess.DEVNULL
    )


def wait_tcp(host, port, timeout=30):
    """
    等待 TCP 端口可连接，用于服务就绪探测。
    """
    for _ in range(timeout):
        try:
            s = socket.create_connection((host, port), timeout=1)
            s.close()
            return True
        except OSError:
            time.sleep(1)
    return False


def ok(msg):
    print(f"  ✅ {msg}")


def warn(msg):
    print(f"  ⚠️  {msg}")


def err(msg):
    print(f"  ❌ {msg}")


def section(title):
    print(f"\n{'─' * 52}\n  {title}\n{'─' * 52}")


# —— 步骤 1：安装 Python 包 ————————————————————————————————————————
section("步骤 1/3  安装 Python 依赖")

for pkg, import_name in [
    ("docker", "docker"),
    ("kubernetes", "kubernetes"),
    ("redis", "redis"),
    ("minio", "minio"),
    ("tabulate", "tabulate"),
    ("pyyaml", "yaml"),
]:
    try:
        __import__(import_name)
        ok(f"{pkg} 已安装")
    except ImportError:
        print(f"  📦 安装 {pkg}...", end=" ", flush=True)
        pip_install(pkg)
        ok(f"{pkg} 完成")


# —— 步骤 2：检测 Docker ——————————————————————————————————————————
section("步骤 2/3  检测 Docker")

if shutil.which("docker"):
    _out, _err, _rc = run("docker info --format '{{.ServerVersion}}'", timeout=20)

    if _rc == 0:
        ok(f"Docker 就绪 (Server: {_out[:30]})")
    else:
        err("Docker 已安装但 daemon 未运行，请启动 Docker Desktop 后重跑本单元格")
        if _err:
            warn(f"Docker 错误信息：{_err[:200]}")
else:
    err("未检测到 Docker，请先安装 Docker Desktop")


# —— 步骤 3：检测 Kubernetes 集群 ——————————————————————————————————
section("步骤 3/3  检测 Kubernetes 集群")

# 先检查 kubectl 是否存在
if not shutil.which("kubectl"):
    err("未检测到 kubectl，任务二将无法完成。请确认 Kubernetes 工具已安装并加入 PATH。")

else:
    # 先检查 kubectl 是否已经能连接集群
    _out, _err, _rc = run("kubectl cluster-info", timeout=20)

    if _rc == 0:
        first_line = _out.splitlines()[0] if _out else "kubectl 已连接集群"
        ok(f"K8s 集群就绪：{first_line[:80]}")

    else:
        # 如果 kubectl 暂时连不上，再检查 minikube 状态
        if shutil.which("minikube"):
            _out2, _err2, _rc2 = run("minikube status", timeout=20)

            # 你的情况就是这里：Minikube 已经 Running
            if (
                _rc2 == 0
                and "host: Running" in _out2
                and "kubelet: Running" in _out2
                and "apiserver: Running" in _out2
            ):
                ok("Minikube 已运行")
                ok("K8s 集群就绪")

                # 再尝试输出节点信息
                _nodes, _nodes_err, _nodes_rc = run("kubectl get nodes", timeout=20)
                if _nodes_rc == 0:
                    print("\n  当前节点：")
                    print(_nodes)
                else:
                    warn("Minikube 显示已运行，但 kubectl get nodes 暂时失败")
                    if _nodes_err:
                        warn(_nodes_err[:200])

            else:
                warn("集群未运行，尝试启动 Minikube")

                # 使用国内镜像参数，避免首次拉镜像过慢
                start_cmd = "minikube start --driver=docker --cpus=2 --memory=4096 --image-mirror-country=cn"

                _out3, _err3, _rc3 = run(start_cmd, timeout=300)

                if _rc3 == 0:
                    ok("Minikube 启动成功")

                    _nodes, _nodes_err, _nodes_rc = run("kubectl get nodes", timeout=30)
                    if _nodes_rc == 0:
                        print("\n  当前节点：")
                        print(_nodes)
                    else:
                        warn("Minikube 启动后，kubectl get nodes 暂时失败")
                        if _nodes_err:
                            warn(_nodes_err[:200])

                else:
                    err("Minikube 启动失败")
                    if _err3:
                        warn(f"错误信息：{_err3[:300]}")
                    elif _out3:
                        warn(f"输出信息：{_out3[:300]}")

        else:
            err("未检测到 Minikube，任务二将无法完成")


print("\n" + "═" * 52 + "\n  环境配置完成，可继续答题\n" + "═" * 52)


────────────────────────────────────────────────────
  步骤 1/3  安装 Python 依赖
────────────────────────────────────────────────────
  ✅ docker 已安装
  ✅ kubernetes 已安装
  ✅ redis 已安装
  ✅ minio 已安装
  ✅ tabulate 已安装
  ✅ pyyaml 已安装

────────────────────────────────────────────────────
  步骤 2/3  检测 Docker
────────────────────────────────────────────────────
  ✅ Docker 就绪 (Server: '29.0.1')

────────────────────────────────────────────────────
  步骤 3/3  检测 Kubernetes 集群
────────────────────────────────────────────────────
  ✅ K8s 集群就绪：Kubernetes control plane is running at https://127.0.0.1:44962

════════════════════════════════════════════════════
  环境配置完成，可继续答题
════════════════════════════════════════════════════


---
## 任务一：Docker 容器生命周期管理（30 分） [第 1-2 讲]

### 任务背景
云计算依赖容器化技术。本任务使用 `docker` Python SDK 启动 Redis 容器，
为任务三提供数据服务。

### 评分细则

| 子项 | 分值 | 关键点 |
|------|------|-------|
| ① 连接 Docker daemon | 5 分 | `docker.from_env()` |
| ② 拉取 redis:7.0 镜像 | 10 分 | `client.images.pull(repo, tag=...)` |
| ③ 启动 Redis 容器（端口 6379） | 15 分 | `ports={'6379/tcp': 6379}` |

---

### 🖊️ 第 1 步：连接 Docker 并拉取镜像（15 分）

> ⚠️ **网络提醒**：若出现 `APIError ... EOF`，通常是 Docker Hub 网络连通性问题，不是你的填空语法错误。
>
> 可按以下方式处理后重跑本单元格：
> 1. 先手动拉镜像站并打标签：`docker pull docker.m.daocloud.io/library/redis:7.0 && docker tag docker.m.daocloud.io/library/redis:7.0 redis:7.0`
> 2. 或在 Docker Desktop 配置镜像加速器（registry-mirrors）后重试。

In [4]:
import docker

# ─────────────────────────────────────────────────────────────────
# 【任务一 · 第1步】连接 Docker daemon + 拉取镜像       （15分）
#
# API 示例：
#   客户端：client = docker.from_env()
#   拉取：  client.images.pull(repo, tag='xxx')
# ─────────────────────────────────────────────────────────────────

def init_docker_and_pull():
    # (1) 获取 Docker 客户端
    client = docker.from_env()

    # 测试 Docker daemon 是否可连接
    try:
        client.ping()
        print("✅ Docker daemon 连接成功")
    except docker.errors.DockerException as e:
        print("❌ Docker Desktop 可能未启动，请先启动 Docker Desktop")
        raise e

    # (2) 优先使用本地 redis:7.0，避免网络不稳定时反复拉取失败
    try:
        client.images.get("redis:7.0")
        print("✅ 本地已有 redis:7.0，跳过拉取")

    except docker.errors.ImageNotFound:
        print("⏳ 本地无 redis:7.0，开始拉取镜像...")

        # 先按题目要求调用官方源 pull
        try:
            client.images.pull("redis", tag="7.0")
            print("✅ redis:7.0 就绪")

        except docker.errors.APIError as e:
            print("⚠️ 官方源拉取失败，尝试镜像站重试...")

            mirrors = [
                ("docker.m.daocloud.io/library/redis", "7.0"),
                ("docker.1ms.run/library/redis", "7.0"),
            ]

            pulled = False

            for repo, tag in mirrors:
                try:
                    print(f"  - 尝试 {repo}:{tag}")

                    client.images.pull(repo, tag=tag)

                    # 把镜像站拉下来的镜像重新打标签为 redis:7.0
                    image = client.images.get(f"{repo}:{tag}")
                    image.tag("redis", tag="7.0")

                    print("  - 已重打标签为 redis:7.0")
                    print("✅ redis:7.0 就绪")

                    pulled = True
                    break

                except docker.errors.APIError as mirror_error:
                    print(f"  - 失败：{str(mirror_error)[:120]}")

            if not pulled:
                print("❌ 自动重试失败，可执行以下命令后重跑当前单元格：")
                print("   docker pull docker.m.daocloud.io/library/redis:7.0")
                print("   docker tag docker.m.daocloud.io/library/redis:7.0 redis:7.0")
                print("   # 或在 Docker Desktop 配置 registry-mirrors 后重试")
                raise e

    return client


docker_client = init_docker_and_pull()

# 验证：打印本地镜像数量
print(f"\n本地镜像总数：{len(docker_client.images.list())}")

✅ Docker daemon 连接成功
⏳ 本地无 redis:7.0，开始拉取镜像...
⚠️ 官方源拉取失败，尝试镜像站重试...
  - 尝试 docker.m.daocloud.io/library/redis:7.0
  - 已重打标签为 redis:7.0
✅ redis:7.0 就绪

本地镜像总数：6


### 🖊️ 第 2 步：启动 Redis 容器（15 分）

**提示**：`containers.run()` 的端口映射参数 `ports` 是一个字典，
键是 **容器内端口/协议**（如 `'6379/tcp'`），值是 **宿主机端口**（如 `6379`）。

In [5]:
# ─────────────────────────────────────────────────────────────────
# 【任务一 · 第2步】启动 Redis 容器                      （15分）
# ─────────────────────────────────────────────────────────────────

def cleanup_if_exists(name):
    # 同名容器若存在则先删除（无需修改）
    try:
        docker_client.containers.get(name).remove(force=True)
        print(f"  🧹 已清理旧容器：{name}")
    except docker.errors.NotFound:
        pass


def start_redis_container():
    cleanup_if_exists('exam-redis')
    return docker_client.containers.run(
        image='redis:7.0',
        name='exam-redis',
        ports={'6379/tcp': 6379},     # 容器 6379 → 宿主机 6379
        detach=True,
    )


redis_ct = start_redis_container()

# —— 等待服务就绪（TCP 探测，最长 30 秒）——
print("\n⏳ 等待 Redis 就绪 ...")
assert wait_tcp('localhost', 6379, 30), "Redis 启动超时"
print("✅ 服务就绪\n")

# —— 打印状态 ——
redis_ct.reload()
tag = redis_ct.image.tags[0] if redis_ct.image.tags else 'N/A'
print(f"  {redis_ct.name:<15} status={redis_ct.status:<10} image={tag}")


⏳ 等待 Redis 就绪 ...
✅ 服务就绪

  exam-redis      status=running    image=docker.1ms.run/library/redis:7.0


---
## 任务二：Kubernetes Deployment + Service（30 分） [第 3-5 讲]

### 任务背景
K8s 是云原生基石。本任务部署一个 Nginx 应用（3 副本），并通过 NodePort Service 暴露。

### 评分细则

| 子项 | 分值 | 关键点 |
|------|------|-------|
| ① Deployment YAML（replicas + image + resources） | 15 分 | cpu 单位 `m`, memory 单位 `Mi` |
| ② Service YAML（NodePort + nodePort 字段） | 5 分 | `type: NodePort`, `nodePort: 30080` |
| ③ Python 客户端读取 Deployment 状态 | 10 分 | `apps_v1.read_namespaced_deployment(...)` |

---

### 🖊️ 第 1 步：编写 Deployment + Service YAML（20 分）

> **填空说明**：YAML 中只需补 5 个值（replicas、image、requests.cpu、Service type、nodePort），
> 其余字段已给定。

In [6]:
# ─────────────────────────────────────────────────────────────────
# 【任务二 · 第1步】编写 Deployment + Service YAML      （20分）
#
# 要求：
#   - Deployment 名称 nginx-exam，副本 3；镜像 nginx:1.25
#   - 资源请求 requests: cpu=50m,  memory=64Mi
#   - 资源上限 limits:   cpu=200m, memory=128Mi
#   - Service 类型 NodePort，端口 80 → targetPort 80 → nodePort 30080
#
# 提示：
#   - CPU 单位 "m" = millicore（千分之一核），50m = 0.05 核
#   - 内存单位 "Mi" = 1024*1024 字节（注意不是 "MB"）
# ─────────────────────────────────────────────────────────────────

K8S_YAML = '''
# ───── Deployment ─────
apiVersion: apps/v1
kind: Deployment
metadata:
  name: nginx-exam
  namespace: default
  labels:
    app: nginx-exam
spec:
  replicas: 3                  # ← 副本数
  selector:
    matchLabels:
      app: nginx-exam
  template:
    metadata:
      labels:
        app: nginx-exam
    spec:
      containers:
        - name: nginx
          image: nginx:1.25    
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 80
          resources:
            requests:
              cpu: 50m         # ← 千分之 50 核
              memory: 64Mi
            limits:
              cpu: 200m
              memory: 128Mi
---
# ───── Service（NodePort） ─────
apiVersion: v1
kind: Service
metadata:
  name: nginx-exam-svc
  namespace: default
spec:
  type: NodePort               # ← Service 类型
  selector:
    app: nginx-exam
  ports:
    - port: 80
      targetPort: 80
      nodePort: 30080          
'''

# —— 写入文件并 dry-run 校验 ——
YAML_FILE = "nginx-exam.yaml"

with open(YAML_FILE, "w", encoding="utf-8") as f:
    f.write(K8S_YAML)

# —— 先在本机拉取，再加载到 Minikube，避免集群内直连 Docker Hub 失败 ——
print("⏳ 本地拉取 nginx:1.25 ...")
_pull_out, _pull_err, _pull_rc = run("docker pull nginx:1.25")

if _pull_rc == 0:
    print("✅ 本地镜像已就绪")
else:
    print("⚠️  本地拉取失败，将继续尝试已有镜像加载：" + (_pull_err[:160] if _pull_err else "unknown error"))

print("⏳ 预加载 nginx:1.25 到 Minikube ...")
_img_out, _img_err, _img_rc = run("minikube image load nginx:1.25")

if _img_rc == 0:
    print("✅ 镜像预加载成功")
else:
    print("⚠️  预加载失败，将继续常规拉取：" + (_img_err[:160] if _img_err else "unknown error"))

_out, _e, _rc = run(f"kubectl apply -f {YAML_FILE} --dry-run=client")

if _rc != 0:
    print("❌ YAML 语法错误：\n" + _e)
else:
    print("✅ dry-run 通过，开始部署...")
    _out2, _e2, _rc2 = run(f"kubectl apply -f {YAML_FILE}")
    print(_out2 if _rc2 == 0 else _e2)

# —— 等待 Pod 就绪（最多 60 秒） ——
print("\n⏳ 等待 Pod 就绪 ...")

for i in range(12):
    _out, _, _ = run("kubectl get deployment nginx-exam -o jsonpath='{.status.readyReplicas}'")
    _ready = int(_out.strip("'") or 0)

    print(f"  [{i * 5:2d}s] 就绪: {_ready}/3", end="\r")

    if _ready == 3:
        print(f"\n✅ 3 个 Pod 已全部就绪")
        break

    time.sleep(5)

else:
    print("\n⚠️  部分 Pod 尚未就绪，但可继续作答第2步")

_out, _, _ = run("kubectl get pods,svc -l app=nginx-exam")
print("\n" + _out)

⏳ 本地拉取 nginx:1.25 ...
✅ 本地镜像已就绪
⏳ 预加载 nginx:1.25 到 Minikube ...
✅ 镜像预加载成功
✅ dry-run 通过，开始部署...
deployment.apps/nginx-exam created
service/nginx-exam-svc created

⏳ 等待 Pod 就绪 ...
  [ 5s] 就绪: 3/3
✅ 3 个 Pod 已全部就绪

NAME                              READY   STATUS    RESTARTS   AGE
pod/nginx-exam-74fd58794c-hwspx   1/1     Running   0          5s
pod/nginx-exam-74fd58794c-qzf68   1/1     Running   0          5s
pod/nginx-exam-74fd58794c-rgg4h   1/1     Running   0          5s


### 🖊️ 第 2 步：Python 客户端读取 Deployment 状态（10 分）

In [7]:
from kubernetes import client, config
from kubernetes.client.exceptions import ApiException

# ─────────────────────────────────────────────────────────────────
# 【任务二 · 第2步】Python 客户端读取 Deployment       （10分）
# ─────────────────────────────────────────────────────────────────

def read_deployment_status():
    try:
        config.load_kube_config()
    except config.ConfigException:
        config.load_incluster_config()

    # (1) 创建 AppsV1Api 客户端
    apps_v1 = client.AppsV1Api()

    # (2) 读取 Deployment 对象（name='nginx-exam', namespace='default'）
    try:
        dep = apps_v1.read_namespaced_deployment(
            name='nginx-exam',
            namespace='default'
        )
    except ApiException as e:
        print(f"❌ 读取失败：{e.status} {e.reason}")
        return None

    # (3) 打印关键字段
    print("=" * 50)
    print("  Deployment 基本信息")
    print("=" * 50)
    print(f"  名称         : {dep.metadata.name}")
    print(f"  命名空间     : {dep.metadata.namespace}")
    print(f"  期望副本数   : {dep.spec.replicas}")
    print(f"  就绪副本数   : {dep.status.ready_replicas or 0}")
    print(f"  可用副本数   : {dep.status.available_replicas or 0}")
    print("=" * 50)

    return dep


dep = read_deployment_status()

  Deployment 基本信息
  名称         : nginx-exam
  命名空间     : default
  期望副本数   : 3
  就绪副本数   : 3
  可用副本数   : 3


---
## 任务三：Redis 数据结构操作（40 分） [第 6-7 讲]

### 任务背景
模拟图书管理系统：Redis 做热度排行（内存型 KV）与用户信息存储（Hash 结构）。

**本任务依赖任务一启动的 Redis（:6379）。**

### 评分细则

| 子项 | 分值 | 关键点 |
|------|------|-------|
| ① Redis String + Hash 操作 | 20 分 | `r.set / r.hset(mapping=...)` |
| ② Redis Sorted Set 排行榜 | 20 分 | `r.zadd / r.zrevrange` |

---

### 🖊️ 第 1 步：Redis String + Hash（20 分）

In [9]:
import redis

# —— 图书数据集（供排行榜任务使用）——
BOOKS = [
    {"title": "云计算概论",        "author": "张三", "year": 2022, "views": 150},
    {"title": "Kubernetes 实战",   "author": "李四", "year": 2023, "views": 320},
    {"title": "Docker 核心技术",   "author": "王五", "year": 2021, "views": 210},
    {"title": "分布式系统",        "author": "赵六", "year": 2024, "views": 480},
    {"title": "Redis 实战",        "author": "陈七", "year": 2023, "views": 90},
]

# ─────────────────────────────────────────────────────────────────
# 【任务三 · 第1步】Redis 数据结构操作                   （20分）
# ─────────────────────────────────────────────────────────────────

def redis_string_and_hash():
    # (1) 连接 Redis（decode_responses=True 是关键）
    r = redis.Redis(
        host='localhost',
        port=6379,
        decode_responses=True
    )

    # 测试 Redis 是否连接成功
    r.ping()
    print("✅ Redis 连接成功")

    # —— String 示例 ——
    r.set('site:name', '云计算实验平台')
    site = r.get('site:name')
    print(f"[String]  site:name = {site}")

    # —— Hash 示例 ——
    # 要求：键 'user:1001'，字段 name=张三 / role=student / grade=90
    r.hset(
        'user:1001',
        mapping={
            'name': '张三',
            'role': 'student',
            'grade': '90'
        }
    )

    user = r.hgetall('user:1001')
    print(f"[Hash]    user:1001 = {user}")

    return r


r = redis_string_and_hash()

✅ Redis 连接成功
[String]  site:name = 云计算实验平台
[Hash]    user:1001 = {'name': '张三', 'role': 'student', 'grade': '90'}


### 🖊️ 第 2 步：Redis Sorted Set 排行榜（20 分）

In [10]:
# ─────────────────────────────────────────────────────────────────
# 【任务三 · 第2步】Redis Sorted Set 排行榜              （20分）
#
# API 示例：
#   r.zadd(key, mapping={'成员': 分数, ...})     # 批量写入
#   r.zrevrange(key, start, end, withscores=True)
#       └── 按分数降序取 [start, end] 位（含两端）；withscores 同时返回分数
#
# 本题：把 BOOKS 列表中的 {title: views} 全部写入 Sorted Set
# ─────────────────────────────────────────────────────────────────

def redis_leaderboard(r):
    r.delete('book:rank')   # 先清空

    # 构造 mapping：{书名 → 浏览数}
    ranking = {b['title']: b['views'] for b in BOOKS}

    # 写入 Sorted Set（使用 zadd 的 mapping 参数）
    r.zadd('book:rank', mapping=ranking)

    # 取前 3 名（降序）
    top3 = r.zrevrange('book:rank', 0, 2, withscores=True)
    return top3


top3 = redis_leaderboard(r)

print("🏆 图书热度 Top 3：")
for i, (title, score) in enumerate(top3, 1):
    print(f"  {i}. {title:<20}  浏览数 = {int(score)}")

🏆 图书热度 Top 3：
  1. 分布式系统                 浏览数 = 480
  2. Kubernetes 实战         浏览数 = 320
  3. Docker 核心技术           浏览数 = 210


---
## 加分项（可选，最多 +10 分）

> 完成以下任一项可获得额外加分，加分项人工评阅。

### 加分项 A：MinIO 对象存储（+5 分） [第 8 讲]

启动 MinIO 容器 → 用 `minio` SDK 创建 Bucket、上传对象、列举对象。

In [11]:
from minio import Minio
from io import BytesIO

# ─────────────────────────────────────────────────────────────────
# 【加分项 A】MinIO 对象存储                             （+5分）
# ─────────────────────────────────────────────────────────────────

# —— 启动 MinIO 容器（已提供，无需修改）——
cleanup_if_exists('exam-minio')

docker_client.containers.run(
    image='minio/minio',
    name='exam-minio',
    ports={'9000/tcp': 9000, '9001/tcp': 9001},
    environment={
        'MINIO_ROOT_USER': 'minioadmin',
        'MINIO_ROOT_PASSWORD': 'minioadmin'
    },
    command='server /data --console-address :9001',
    detach=True,
)

assert wait_tcp('localhost', 9000, 30), "MinIO 启动超时"
print("✅ MinIO 就绪 (endpoint: localhost:9000)")


# —— 你的作答 ——
def minio_operations():
    # (1) 创建 MinIO 客户端
    mc = Minio(
        'localhost:9000',
        access_key='minioadmin',
        secret_key='minioadmin',
        secure=False
    )

    # (2) 创建 Bucket（先检查是否已存在）
    if not mc.bucket_exists('exam-bucket'):
        mc.make_bucket('exam-bucket')

    print("✅ Bucket 'exam-bucket' 就绪")

    # (3) 上传 hello.txt（内容："Hello Cloud Computing Midterm!"）
    data = b"Hello Cloud Computing Midterm!"

    mc.put_object(
        'exam-bucket',
        'hello.txt',
        BytesIO(data),
        length=len(data)
    )

    print("✅ hello.txt 上传成功")

    # 列举（已提供）
    print("\n📂 Bucket 内容：")
    for obj in mc.list_objects('exam-bucket'):
        print(f"  - {obj.object_name}  ({obj.size} 字节)")


minio_operations()

✅ MinIO 就绪 (endpoint: localhost:9000)
✅ Bucket 'exam-bucket' 就绪
✅ hello.txt 上传成功

📂 Bucket 内容：
  - hello.txt  (30 字节)


### 加分项 B：Docker Compose YAML（+3 分） [第 3 讲]

补全 Compose 文件：编排 MongoDB 7 + Redis 7.0两个服务，共用自定义 bridge 网络。

In [13]:
# ─────────────────────────────────────────────────────────────────
# 【加分项 B】Docker Compose 多容器编排                  （+3分）
#
# 考点：
#   - services 下的镜像字段使用完整的 image:tag
#   - networks 定义自定义网络，driver: bridge
# ─────────────────────────────────────────────────────────────────

COMPOSE_YAML = '''
version: "3.8"
services:
  mongodb:
    image: mongo:6.0
    container_name: compose-mongo
    ports:
      - "27017:27017"
    volumes:
      - mongo-data:/data/db
    networks:
      - app-net

  redis:
    image: redis:7.0
    container_name: compose-redis
    ports:
      - "6379:6379"
    networks:
      - app-net

volumes:
  mongo-data: {}

networks:
  app-net:
    driver: bridge
'''

# —— 结构校验（已提供） ——
import yaml

try:
    parsed = yaml.safe_load(COMPOSE_YAML)
    assert 'services' in parsed and len(parsed['services']) == 2
    assert parsed['networks']['app-net']['driver'] == 'bridge'
    print("✅ Compose YAML 结构校验通过")
    print(f"   services: {list(parsed['services'].keys())}")
    print(f"   networks: {list(parsed['networks'].keys())}")
except Exception as e:
    print(f"❌ YAML 结构错误：{e}")

✅ Compose YAML 结构校验通过
   services: ['mongodb', 'redis']
   networks: ['app-net']


### 加分项 C：熔断器模式（+2 分） [第 3 讲]

实现极简熔断器：连续失败 ≥ 阈值时进入 OPEN 状态，之后直接拒绝请求。

In [14]:
# ─────────────────────────────────────────────────────────────────
# 【加分项 C】Circuit Breaker 熔断器                     （+2分）
#
# 状态机：
#   CLOSED  ─ 失败数 >= threshold ─▶  OPEN
#   CLOSED  ─ 调用成功 ─▶  CLOSED（failures 归零）
#   OPEN    ─ 任何调用 ─▶  直接抛错（快速失败）
# ─────────────────────────────────────────────────────────────────

class CircuitBreaker:
    def __init__(self, threshold=3):
        self.threshold = threshold
        self.failures = 0
        self.state = 'CLOSED'

    def call(self, func):
        # 若已熔断，直接抛错（快速失败）
        if self.state == 'OPEN':
            raise RuntimeError("Circuit breaker is OPEN")

        try:
            result = func()
            self.failures = 0            # 成功时重置失败计数
            return result
        except Exception:
            self.failures += 1
            if self.failures >= self.threshold:
                self.state = 'OPEN'      # 超过阈值，切换到 OPEN
            raise


# —— 自验（已提供） ——
cb = CircuitBreaker(threshold=3)

def always_fail():
    raise ValueError("模拟故障")


triggered_open = False

for i in range(5):
    try:
        cb.call(always_fail)
    except RuntimeError as e:
        print(f"第 {i+1} 次：熔断器拒绝 — {e}")
        triggered_open = True
        break
    except ValueError:
        print(f"第 {i+1} 次：调用失败 (failures={cb.failures}, state={cb.state})")

print(f"\n{'✅' if triggered_open else '❌'} 测试：连续失败触发熔断，state = {cb.state}")

第 1 次：调用失败 (failures=1, state=CLOSED)
第 2 次：调用失败 (failures=2, state=CLOSED)
第 3 次：调用失败 (failures=3, state=OPEN)
第 4 次：熔断器拒绝 — Circuit breaker is OPEN

✅ 测试：连续失败触发熔断，state = OPEN


---
## 自动评分检查

运行下方单元格查看必做题自动评分（满分 100），加分项由教师人工评阅。

In [12]:
# AUTOSCORE_BEGIN
score = 0
results = []

def check(item, cond, pts, hint=""):
    global score
    _ok = bool(cond)
    if _ok: score += pts
    results.append(("✅" if _ok else "❌", item, pts, _ok, hint))

# ══ 任务一（30 分）═══════════════════════════════════════════════
try:
    _tags = [t for img in docker_client.images.list() for t in (img.tags or [])]
    check("① Docker 客户端连通",
          docker_client is not None, 5)
    check("② redis:7.0 已拉取",
          any('redis:7.0' in t for t in _tags), 10,
          "检查 pull 调用是否完整")
    _r = docker_client.containers.get('exam-redis'); _r.reload()
    check("③ exam-redis 容器运行中",
          _r.status == 'running' and wait_tcp('localhost', 6379, 5), 15,
          "端口映射 6379/tcp → 6379")
except Exception as e:
    for lbl, pts in [("① Docker 客户端连通", 5),
                     ("② redis:7.0 已拉取", 10),
                     ("③ exam-redis 容器运行中", 15)]:
        check(lbl, False, pts, str(e)[:60])

# ══ 任务二（30 分）═══════════════════════════════════════════════
try:
    _y = K8S_YAML
    check("④ Deployment（replicas=3 + image + resources）",
          "kind: Deployment" in _y and "replicas: 3" in _y
          and "nginx:1.25" in _y and "50m" in _y
          and "64Mi" in _y and "200m" in _y and "128Mi" in _y, 15,
          "填空：replicas=3 / image=nginx:1.25 / requests.cpu=50m")
    check("⑤ Service（NodePort + nodePort:30080）",
          "type: NodePort" in _y and "nodePort: 30080" in _y, 5,
          "填空：type=NodePort / nodePort=30080")
except Exception as e:
    check("④ Deployment YAML 检查", False, 15, str(e)[:60])
    check("⑤ Service YAML 检查",  False, 5,  str(e)[:60])

try:
    _apps = client.AppsV1Api()
    _dep = _apps.read_namespaced_deployment(name='nginx-exam', namespace='default')
    check("⑥ Python 客户端读取 Deployment（replicas=3）",
          _dep is not None and _dep.spec.replicas == 3, 10,
          "read_namespaced_deployment 或 print 字段错误")
except Exception as e:
    check("⑥ Python 客户端读取 Deployment（replicas=3）", False, 10, str(e)[:60])

# ══ 任务三（40 分）═══════════════════════════════════════════════
try:
    check("⑦ Redis String + Hash 写入",
          r.get('site:name') == '云计算实验平台'
          and r.hgetall('user:1001').get('name') == '张三', 20,
          "hset mapping 字段检查")
    _top = r.zrevrange('book:rank', 0, 2, withscores=True)
    check("⑧ Redis Sorted Set Top 3（第一名 views=480）",
          len(_top) == 3 and int(_top[0][1]) == 480, 20,
          "zadd mapping 未正确执行")
except Exception as e:
    for lbl, pts in [("⑦ Redis String + Hash 写入", 20),
                     ("⑧ Redis Sorted Set Top 3（第一名 views=480）", 20)]:
        check(lbl, False, pts, str(e)[:60])

# ══ 输出报告 ═══════════════════════════════════════════════════════
SEP = "═" * 68
print(f"\n{SEP}")
print(f"  期中考试 · 自动评分报告（必做题满分 100）")
print(f"{SEP}")
print(f"  {'检查项':<44} {'分值':>4}   结果")
print(f"  {'─'*42:<44} {'─'*4}   ────")
for status, item, pts, _ok, hint in results:
    note = f"  ← {hint}" if not _ok and hint else ""
    print(f"  {status} {item:<42} {pts:>3}分   {'得分' if _ok else '未得分'}{note}")
print(f"{SEP}")
print(f"  必做题合计：{score} / 100 分")
print(f"  加分项 A / B / C  需教师人工评阅，最多 +10 分")
print(f"{SEP}")
# AUTOSCORE_END



════════════════════════════════════════════════════════════════════
  期中考试 · 自动评分报告（必做题满分 100）
════════════════════════════════════════════════════════════════════
  检查项                                            分值   结果
  ──────────────────────────────────────────   ────   ────
  ✅ ① Docker 客户端连通                               5分   得分
  ✅ ② redis:7.0 已拉取                             10分   得分
  ✅ ③ exam-redis 容器运行中                          15分   得分
  ✅ ④ Deployment（replicas=3 + image + resources）  15分   得分
  ✅ ⑤ Service（NodePort + nodePort:30080）         5分   得分
  ✅ ⑥ Python 客户端读取 Deployment（replicas=3）       10分   得分
  ✅ ⑦ Redis String + Hash 写入                    20分   得分
  ✅ ⑧ Redis Sorted Set Top 3（第一名 views=480）     20分   得分
════════════════════════════════════════════════════════════════════
  必做题合计：100 / 100 分
  加分项 A / B / C  需教师人工评阅，最多 +10 分
════════════════════════════════════════════════════════════════════


---
## 清理资源（交卷前请先截图保存得分报告）

In [ ]:
# ⚠️ 请先截图保存评分报告，再运行此单元格

# 清理 K8s 资源
run("kubectl delete -f /tmp/nginx-exam.yaml --ignore-not-found=true")
print("✅ K8s 资源已清理")

# 清理 Docker 容器
for name in ['exam-redis', 'exam-minio']:
    try:
        docker_client.containers.get(name).remove(force=True)
        print(f"✅ 容器 {name} 已删除")
    except Exception:
        pass

print("\n🎉 考试结束，请将本 notebook 另存为 [学号_姓名_midterm.ipynb] 并提交")


---
**提交格式**：另存为 `学号_姓名_midterm.ipynb` 后上传至课程邮箱。

*云计算技术课程组 · 2026 春期中考试*